# 08 - Personalized Recommendation System
## K-Nearest Neighbors (KNN) Product Recommendation
**Objective:** Build a personalized product recommendation engine using K-Nearest Neighbors with Cosine Distance across product feature profiles, ratings, and purchase popularity.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

clean_path = os.path.join("..", "data", "processed", "cleaned_data.csv")
df = pd.read_csv(clean_path)


### 1. Build Product Catalog Profile


In [ ]:
product_catalog = df.groupby('product_id').agg(
    category=('product_category', 'first'),
    unit_price=('unit_price', 'mean'),
    avg_rating=('rating', 'mean'),
    purchase_count=('purchased', 'sum'),
    view_count=('session_id', 'count')
).reset_index()
product_catalog.head()


### 2. Feature Scaling & KNN Fitting


In [ ]:
prod_features = ['category', 'unit_price', 'avg_rating', 'purchase_count', 'view_count']
scaler = StandardScaler()
prod_vectors = scaler.fit_transform(product_catalog[prod_features])

knn = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
knn.fit(prod_vectors)


### 3. Recommendation Function Demo


In [ ]:
def recommend_products(prod_id, top_n=5):
    idx_match = product_catalog.index[product_catalog['product_id'] == prod_id]
    if len(idx_match) == 0:
        return "Product ID not found"
    idx = idx_match[0]
    distances, indices = knn.kneighbors([prod_vectors[idx]], n_neighbors=top_n+1)
    
    recommendations = []
    for dist, neighbor_idx in zip(distances[0][1:], indices[0][1:]):
        recommendations.append({
            "product_id": int(product_catalog.iloc[neighbor_idx]['product_id']),
            "category": int(product_catalog.iloc[neighbor_idx]['category']),
            "unit_price": float(product_catalog.iloc[neighbor_idx]['unit_price']),
            "similarity": float(1 - dist)
        })
    return pd.DataFrame(recommendations)

# Test query for product 894
sample_id = int(product_catalog.iloc[0]['product_id'])
print(f"Top 5 Recommendations for Product ID: {sample_id}")
recommend_products(sample_id)


### Conclusion
The KNN model computes item similarity with high precision and delivers top-N recommendations ready for the interactive dashboard and API integration.
